# VAE 从零实现：购物篮潜变量与重参数化

## 面试问题

面试时我会先区分普通自编码器和 VAE：编码器输出的是均值与对数方差，重参数化把随机性写成 `mu + std × epsilon`，从而让梯度穿过采样。目标函数同时包含重建损失和对标准正态先验的 KL 正则；KL 太强会后验坍塌，太弱则潜空间出现空洞。评估不能只看总 loss，还要分别观察重建误差、KL、潜变量和生成样本。稳定实现通常预测 logvar，并用 `exp(0.5 × logvar)` 得到标准差。下面用 12 个可读购物篮画像比较全局均值基线，手写 VAE、反向传播，并复现把方差误当标准差的 bug。

## 真实案例

每条记录是一个脱敏会员近月在咖啡与运动六类商品上的归一化购买强度，包含咖啡偏好、运动偏好和混合生活方式三组画像。数据为结构真实的离线教学样本，只有 12 条且在全数据上拟合，不能代表生成质量或新客泛化。

本实验是为了看清机制而构造的离线小样本，不代表线上收益，也不能外推到开放分布。

In [1]:
import math  # 导入对数函数以构造可验证的方差案例。
import torch  # 导入 PyTorch 以实现 VAE 和随机采样。
from torch import nn  # 导入神经网络基础层。
import torch.nn.functional as F  # 导入激活函数和重建损失。
torch.manual_seed(30)  # 固定随机种子以复现实验输出。
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小实验运行。
feature_names = ["咖啡豆", "滤纸", "马克杯", "瑜伽垫", "弹力带", "运动水壶"]  # 定义六个可读购物强度字段。
profile_ids = ["咖啡客-1", "咖啡客-2", "咖啡客-3", "咖啡客-4", "运动客-1", "运动客-2", "运动客-3", "运动客-4", "混合客-1", "混合客-2", "混合客-3", "混合客-4"]  # 定义十二个脱敏会员画像编号。
segment_names = ["咖啡"] * 4 + ["运动"] * 4 + ["混合"] * 4  # 保存用于解释而不参与训练的画像分组。
baskets = torch.tensor([[1.0, 0.8, 0.7, 0.0, 0.1, 0.2], [0.9, 1.0, 0.6, 0.1, 0.0, 0.3], [0.8, 0.7, 1.0, 0.0, 0.2, 0.2], [1.0, 0.9, 0.8, 0.1, 0.1, 0.2], [0.0, 0.1, 0.2, 1.0, 0.8, 0.7], [0.1, 0.0, 0.2, 0.9, 1.0, 0.6], [0.2, 0.1, 0.3, 0.8, 0.7, 1.0], [0.1, 0.2, 0.2, 1.0, 0.9, 0.8], [0.7, 0.5, 0.6, 0.6, 0.5, 0.7], [0.6, 0.7, 0.5, 0.5, 0.7, 0.6], [0.5, 0.6, 0.7, 0.7, 0.6, 0.5], [0.7, 0.6, 0.5, 0.6, 0.7, 0.6]], dtype=torch.float32)  # 保存十二条六维购物篮强度。
print("画像       分群   " + "  ".join(feature_names))  # 打印输入预览表头。
for index, profile_id in enumerate(profile_ids):  # 逐画像展示真实语义字段。
    values = "  ".join(f"{value:.1f}" for value in baskets[index].tolist())  # 把六维强度转成可读字符串。
    print(f"{profile_id:<8}  {segment_names[index]}    {values}")  # 输出当前画像分群和购物强度。
print(f"输入形状={tuple(baskets.shape)}，取值范围=[{baskets.min().item():.1f}, {baskets.max().item():.1f}]")  # 汇总样本规模和数值边界。

画像       分群   咖啡豆  滤纸  马克杯  瑜伽垫  弹力带  运动水壶
咖啡客-1     咖啡    1.0  0.8  0.7  0.0  0.1  0.2
咖啡客-2     咖啡    0.9  1.0  0.6  0.1  0.0  0.3
咖啡客-3     咖啡    0.8  0.7  1.0  0.0  0.2  0.2
咖啡客-4     咖啡    1.0  0.9  0.8  0.1  0.1  0.2
运动客-1     运动    0.0  0.1  0.2  1.0  0.8  0.7
运动客-2     运动    0.1  0.0  0.2  0.9  1.0  0.6
运动客-3     运动    0.2  0.1  0.3  0.8  0.7  1.0
运动客-4     运动    0.1  0.2  0.2  1.0  0.9  0.8
混合客-1     混合    0.7  0.5  0.6  0.6  0.5  0.7
混合客-2     混合    0.6  0.7  0.5  0.5  0.7  0.6
混合客-3     混合    0.5  0.6  0.7  0.7  0.6  0.5
混合客-4     混合    0.7  0.6  0.5  0.6  0.7  0.6
输入形状=(12, 6)，取值范围=[0.0, 1.0]


## 基线：所有会员都重建为全局均值

最便宜的压缩器不学习个体差异，直接用六个字段的全局均值重建每条记录。VAE 必须在同一均方误差口径上优于这个基线，才说明潜变量携带了画像信息。

In [2]:
global_mean = baskets.mean(dim=0, keepdim=True)  # 计算所有会员共享的六维平均购物篮。
baseline_reconstruction = global_mean.repeat(len(baskets), 1)  # 用同一个平均向量重建每条记录。
baseline_sample_mse = ((baseline_reconstruction - baskets) ** 2).mean(dim=1)  # 计算每条画像的均值基线误差。
baseline_mse = baseline_sample_mse.mean().item()  # 汇总全局均值重建的平均误差。
print(f"全局均值向量={[round(value, 3) for value in global_mean.squeeze(0).tolist()]}")  # 展示基线实际输出的六维向量。
print("画像       均值基线 MSE")  # 打印逐画像基线误差表头。
for index, profile_id in enumerate(profile_ids):  # 遍历所有画像观察基线误差。
    print(f"{profile_id:<8}  {baseline_sample_mse[index].item():.4f}")  # 输出当前画像的重建均方误差。
print(f"均值基线平均 MSE={baseline_mse:.4f}")  # 汇总同数据上的基线指标。

全局均值向量=[0.55, 0.517, 0.525, 0.525, 0.525, 0.533]
画像       均值基线 MSE
咖啡客-1     0.1468
咖啡客-2     0.1454
咖啡客-3     0.1357
咖啡客-4     0.1496
运动客-1     0.1518
运动客-2     0.1576
运动客-3     0.1118
运动客-4     0.1410
混合客-1     0.0104
混合客-2     0.0121
混合客-3     0.0129
混合客-4     0.0118
均值基线平均 MSE=0.0989


## 手写核心：`mu`、`logvar`、重参数化与 KL

编码器输出二维高斯分布参数。训练时采样，评估重建时令 `epsilon=0` 使用均值，避免随机噪声掩盖模型是否学会了结构。

In [3]:
class BasketVAE(nn.Module):  # 定义从编码分布到解码重建的完整 VAE。
    def __init__(self, input_dim, hidden_dim, latent_dim):  # 根据输入、隐藏和潜变量维度创建网络。
        super().__init__()  # 初始化父类以注册全部参数。
        self.encoder = nn.Linear(input_dim, hidden_dim)  # 把六维购物篮映射到隐藏表示。
        self.mean_head = nn.Linear(hidden_dim, latent_dim)  # 预测潜变量高斯分布的均值。
        self.log_variance_head = nn.Linear(hidden_dim, latent_dim)  # 预测潜变量方差的对数。
        self.decoder_hidden = nn.Linear(latent_dim, hidden_dim)  # 把二维潜变量映射回隐藏空间。
        self.decoder_output = nn.Linear(hidden_dim, input_dim)  # 输出六维购物强度 logits。
    def encode(self, inputs):  # 计算后验分布的均值和对数方差。
        hidden = torch.relu(self.encoder(inputs))  # 提取购物篮的非线性隐藏表示。
        mean = self.mean_head(hidden)  # 输出每条画像的二维潜变量均值。
        log_variance = self.log_variance_head(hidden)  # 输出每条画像的二维对数方差。
        return mean, log_variance  # 返回可解释的后验参数。
    def reparameterize(self, mean, log_variance, sample=True):  # 用可导方式从后验分布取样。
        standard_deviation = torch.exp(0.5 * log_variance)  # 把 log 方差正确转换为标准差。
        noise = torch.randn_like(standard_deviation) if sample else torch.zeros_like(standard_deviation)  # 训练时采样而评估时取均值。
        latent = mean + standard_deviation * noise  # 把随机性移到参数之外以保留梯度。
        return latent, standard_deviation  # 返回潜变量与对应标准差。
    def decode(self, latent):  # 把潜变量解码为六维购物强度。
        hidden = torch.relu(self.decoder_hidden(latent))  # 从二维潜变量恢复非线性隐藏表示。
        return torch.sigmoid(self.decoder_output(hidden))  # 用 sigmoid 把重建限制在零到一。
    def forward(self, inputs, sample=True):  # 完成编码、重参数化和解码的完整前向过程。
        mean, log_variance = self.encode(inputs)  # 计算后验均值与对数方差。
        latent, standard_deviation = self.reparameterize(mean, log_variance, sample=sample)  # 执行可导随机采样。
        reconstruction = self.decode(latent)  # 把样本潜变量解码成购物篮。
        return reconstruction, mean, log_variance, latent, standard_deviation  # 暴露所有关键中间量。
vae = BasketVAE(input_dim=6, hidden_dim=12, latent_dim=2)  # 实例化二维潜空间的手写 VAE。
print(vae)  # 展示编码器与解码器的真实层次。
print(f"可训练参数量={sum(parameter.numel() for parameter in vae.parameters())}")  # 输出教学模型参数规模。

BasketVAE(
  (encoder): Linear(in_features=6, out_features=12, bias=True)
  (mean_head): Linear(in_features=12, out_features=2, bias=True)
  (log_variance_head): Linear(in_features=12, out_features=2, bias=True)
  (decoder_hidden): Linear(in_features=2, out_features=12, bias=True)
  (decoder_output): Linear(in_features=12, out_features=6, bias=True)
)
可训练参数量=250


In [4]:
optimizer = torch.optim.Adam(vae.parameters(), lr=0.02)  # 创建优化器更新编码器和解码器。
beta = 0.002  # 设置较小 KL 权重以兼顾小数据重建与潜空间正则。
loss_trace = []  # 保存总损失轨迹。
reconstruction_trace = []  # 保存重建损失轨迹。
kl_trace = []  # 保存 KL 正则轨迹。
first_gradient_norm = 0.0  # 预留首轮均值头梯度范数。
for epoch in range(601):  # 在十二条教学画像上执行六百零一次更新。
    optimizer.zero_grad()  # 清空上一轮累计梯度。
    reconstruction, mean, log_variance, latent, standard_deviation = vae(baskets, sample=True)  # 运行含重参数采样的完整前向传播。
    reconstruction_loss = F.mse_loss(reconstruction, baskets)  # 计算六维购物强度的平均重建误差。
    kl_loss = -0.5 * torch.mean(1.0 + log_variance - mean.pow(2) - log_variance.exp())  # 手写对标准正态先验的 KL 散度。
    loss = reconstruction_loss + beta * kl_loss  # 按 beta 权重合并重建项与 KL 项。
    loss.backward()  # 通过重参数化路径反向传播到编码器。
    if epoch == 0:  # 首轮记录均值头的真实梯度规模。
        first_gradient_norm = vae.mean_head.weight.grad.norm().item()  # 读取后验均值参数的梯度范数。
    optimizer.step()  # 根据当前梯度更新 VAE 参数。
    loss_trace.append(loss.item())  # 保存当前总损失。
    reconstruction_trace.append(reconstruction_loss.item())  # 保存当前重建损失。
    kl_trace.append(kl_loss.item())  # 保存当前 KL 散度。
    if epoch in [0, 50, 200, 600]:  # 选择关键轮次输出分项训练轨迹。
        print(f"epoch={epoch:03d} total={loss.item():.4f} recon={reconstruction_loss.item():.4f} KL={kl_loss.item():.4f}")  # 展示总损失不是单一黑盒数字。
vae.eval()  # 切换到评估模式生成确定性重建。
with torch.no_grad():  # 关闭评估阶段的梯度记录。
    vae_reconstruction, latent_mean, latent_log_variance, deterministic_latent, latent_std = vae(baskets, sample=False)  # 使用后验均值执行确定性重建。
vae_sample_mse = ((vae_reconstruction - baskets) ** 2).mean(dim=1)  # 计算每条画像的 VAE 重建误差。
vae_mse = vae_sample_mse.mean().item()  # 汇总同数据上的平均重建 MSE。
print(f"首轮均值头梯度范数={first_gradient_norm:.6f}")  # 输出非零梯度证明随机采样路径可导。
print(f"首条画像 mu={latent_mean[0].tolist()}，logvar={latent_log_variance[0].tolist()}，std={latent_std[0].tolist()}")  # 展示编码分布的关键中间参数。

epoch=000 total=0.1023 recon=0.1021 KL=0.0575
epoch=050 total=0.0151 recon=0.0092 KL=2.9385


epoch=200 total=0.0110 recon=0.0081 KL=1.4479


epoch=600 total=0.0089 recon=0.0065 KL=1.2267
首轮均值头梯度范数=0.005373
首条画像 mu=[-1.396536111831665, 0.026148155331611633]，logvar=[-3.567978858947754, -0.09319797903299332]，std=[0.16796672344207764, 0.9544700980186462]


## 结果解读：潜变量与逐画像重建

二维 `mu` 是每条画像在潜空间中的位置；重建误差则与全局均值基线同口径。潜变量分组只用于解释这个受控数据，不能声称自动发现了真实用户群。

In [5]:
print("画像       分群   mu-1     mu-2     均值MSE  VAE-MSE")  # 打印潜变量与重建结果表头。
for index, profile_id in enumerate(profile_ids):  # 逐画像展示可读潜变量和误差。
    print(f"{profile_id:<8}  {segment_names[index]}   {latent_mean[index, 0]:>7.3f}  {latent_mean[index, 1]:>7.3f}   {baseline_sample_mse[index]:.4f}   {vae_sample_mse[index]:.4f}")  # 输出当前画像的二维位置和同口径误差。
print(f"平均重建 MSE：全局均值={baseline_mse:.4f}，手写 VAE={vae_mse:.4f}")  # 汇总基线和 VAE 的重建表现。
print(f"咖啡组潜变量中心={latent_mean[:4].mean(dim=0).tolist()}，运动组中心={latent_mean[4:8].mean(dim=0).tolist()}")  # 展示不同业务画像在潜空间的中心位置。

画像       分群   mu-1     mu-2     均值MSE  VAE-MSE
咖啡客-1     咖啡    -1.397    0.026   0.1468   0.0026
咖啡客-2     咖啡    -1.487    0.052   0.1454   0.0116
咖啡客-3     咖啡    -1.236   -0.021   0.1357   0.0162
咖啡客-4     咖啡    -1.497    0.023   0.1496   0.0020
运动客-1     运动     1.594    0.026   0.1518   0.0030
运动客-2     运动     1.846   -0.035   0.1576   0.0029
运动客-3     运动     1.114    0.035   0.1118   0.0028
运动客-4     运动     1.495    0.022   0.1410   0.0028
混合客-1     混合    -0.100   -0.016   0.0104   0.0073
混合客-2     混合    -0.080    0.023   0.0121   0.0056
混合客-3     混合     0.043   -0.023   0.0129   0.0088
混合客-4     混合     0.067   -0.019   0.0118   0.0031
平均重建 MSE：全局均值=0.0989，手写 VAE=0.0057
咖啡组潜变量中心=[-1.4041330814361572, 0.020063910633325577]，运动组中心=[1.5121395587921143, 0.01195068284869194]


## 失败案例：把方差当成标准差

当 `logvar = log(4)` 时，目标方差为 4、目标标准差为 2。错误写法 `exp(logvar)` 会得到标准差 4；正确写法必须是 `exp(0.5×logvar)`。下面用固定随机样本直接量化误差。

In [6]:
torch.manual_seed(300)  # 为重参数化公式对照固定独立随机种子。
demo_log_variance = torch.full((20000,), math.log(4.0))  # 构造方差恰为四的大样本后验。
demo_noise = torch.randn_like(demo_log_variance)  # 生成共享的标准正态噪声以公平比较公式。
wrong_samples = torch.exp(demo_log_variance) * demo_noise  # 复现把方差直接当标准差的错误写法。
fixed_samples = torch.exp(0.5 * demo_log_variance) * demo_noise  # 使用平方根关系得到正确标准差。
target_standard_deviation = 2.0  # 记录方差四对应的理论标准差。
wrong_empirical_std = wrong_samples.std().item()  # 估计错误公式产生的经验标准差。
fixed_empirical_std = fixed_samples.std().item()  # 估计正确公式产生的经验标准差。
wrong_std_error = abs(wrong_empirical_std - target_standard_deviation)  # 计算错误公式到理论值的偏差。
fixed_std_error = abs(fixed_empirical_std - target_standard_deviation)  # 计算正确公式到理论值的偏差。
print(f"目标标准差={target_standard_deviation:.1f}，错误 exp(logvar) 的经验标准差={wrong_empirical_std:.3f}")  # 展示错误公式放大随机性的程度。
print(f"目标标准差={target_standard_deviation:.1f}，修复 exp(0.5*logvar) 的经验标准差={fixed_empirical_std:.3f}")  # 展示正确公式接近理论值。
print("修复结论：网络预测 log 方差时，采样前必须乘 0.5 再取指数。")  # 总结重参数化实现的关键细节。

目标标准差=2.0，错误 exp(logvar) 的经验标准差=4.015
目标标准差=2.0，修复 exp(0.5*logvar) 的经验标准差=2.008
修复结论：网络预测 log 方差时，采样前必须乘 0.5 再取指数。


## 生产差距

真实 VAE 需要训练/验证切分、离散商品分布对应的似然、KL warm-up、后验坍塌监控、潜空间漂移和生成安全审核。这里用 MSE 与 sigmoid 只是教学简化；若字段是计数或类别，应换成匹配数据分布的解码头和损失。

## 最小回归测试

In [7]:
assert len(profile_ids) >= 6  # 保证案例包含足够多的真实语义购物画像。
assert loss_trace[-1] < loss_trace[0]  # 保证真实反向传播使 VAE 总损失下降。
assert first_gradient_norm > 0.0  # 保证梯度穿过重参数化路径到达均值头。
assert vae_mse < baseline_mse  # 保证 VAE 在同一重建指标上优于全局均值。
assert torch.isfinite(latent_mean).all() and torch.isfinite(latent_log_variance).all()  # 保证潜变量分布参数没有数值异常。
assert fixed_std_error < wrong_std_error  # 保证正确标准差公式比错误公式更接近理论值。
assert fixed_std_error < 0.05  # 保证足量样本下正确经验标准差接近二。
print("回归测试通过：重建、可导采样和标准差公式均符合预期。")  # 输出集中断言的最终验收结论。

回归测试通过：重建、可导采样和标准差公式均符合预期。
